In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorNeuropixelsProjectCache

In [ ]:
session_id = 1119946360

In [ ]:
cache = VisualBehaviorNeuropixelsProjectCache.from_s3_cache(r"E:\jerry\vbn_s3_cache")
session = cache.get_ecephys_session(1119946360)

In [ ]:
import numpy as np
import shutil
from pynwb import NWBHDF5IO
import blosc2

In [ ]:
path = r"E:\jerry\vbn_s3_cache\visual-behavior-neuropixels-0.4.0\behavior_ecephys_sessions\1119946360\ecephys_session_1119946360.nwb"
io = NWBHDF5IO(path, mode="r") # type: ignore
nwbfile = io.read()

In [ ]:
units = session.get_units()

In [ ]:
# IMPORTANT
# Get the spike times
units_nwb = nwbfile.units.to_dataframe()
units_nwb_spiketimes = units_nwb.spike_times
spike_times_manual = {}
for unit_id, spike_times in units_nwb_spiketimes.items():
    spike_times_manual[unit_id] = np.array(spike_times)
spike_times = session.spike_times
# check if all units are present
print(list(spike_times_manual.keys()) == list(spike_times.keys()))
# for each unit, check if the spike times match
for unit_id in tqdm(list(spike_times_manual.keys())):
    assert (spike_times_manual[unit_id] == spike_times[unit_id]).all()

In [ ]:
# IMPORTANT
# Get the probes table
num_electrodes = len(list(nwbfile.electrode_groups.keys()))
probes_nwb_dict = {}
counter = 0
for probe_name, probe in nwbfile.electrode_groups.items():
    probes_nwb_dict[counter] = {
        'id': probe.probe_id,
        'name': probe_name,
        'location': probe.location,
        'sampling_rate': probe.device.sampling_rate,
        'lfp_sampling_rate': probe.lfp_sampling_rate*2,
        'has_lfp_data': probe.has_lfp_data,
    }
    counter += 1
probes_nwb = pd.DataFrame.from_dict(probes_nwb_dict, orient='index')
probes_nwb = probes_nwb.set_index('id')
print(probes_nwb.equals(session.probes))

In [ ]:
channels = session.get_channels()

In [ ]:
# IMPORTANT
# Get the channel table
# First half
channel_df = pd.read_csv(r"E:\jerry\vbn_s3_cache\visual-behavior-neuropixels-0.5.0\project_metadata\channels.csv")
channel_df.set_index('ecephys_channel_id', inplace=True)
channel_df = channel_df[(channel_df['ecephys_session_id'] == session_id) & (channel_df['valid_data'] == True)]
channels_df_selected = channel_df[['anterior_posterior_ccf_coordinate',
                                   'dorsal_ventral_ccf_coordinate',
                                   'left_right_ccf_coordinate',
                                   'structure_acronym']]
channels_df_selected.reset_index(inplace=True)
channels_df_selected = channels_df_selected.rename(columns={'ecephys_channel_id':'id'})
channels_df_selected = channels_df_selected.set_index('id')
# Second half
channels_nwb = nwbfile.electrodes.to_dataframe()
channels_nwb = channels_nwb[channels_nwb.valid_data == True]
channels_nwb_selected = channels_nwb[['filtering',
                                      'probe_channel_number', 
                                      'probe_horizontal_position',
                                      'probe_id',
                                      'probe_vertical_position']]
channels_nwb_combined = pd.concat([channels_nwb_selected, channels_df_selected],axis=1)
channels_nwb_combined_sorted = channels_nwb_combined.reindex(sorted(channels_nwb_combined.columns), axis=1)
print(channels_nwb_combined_sorted.equals(channels))